# 🔬 반도체 웨이퍼 불량 탐지 — Deep Learning

**WM-811K 데이터셋** 기반 결함 패턴 8종 분류

| 클래스 | 설명 | 회전 불변? |
|---|---|---|
| Center | 중앙 결함 | ✅ |
| Donut | 도넛 링 결함 | ✅ |
| Edge-Loc | 가장자리 국소 결함 | ⚠️ 방향 있음 |
| Edge-Ring | 가장자리 전체 링 | ✅ |
| Loc | 특정 위치 클러스터 | ⚠️ 위치 있음 |
| Near-full | 거의 전체 결함 | ✅ |
| Random | 무작위 산포 | ✅ |
| Scratch | 스크래치 선 | ⚠️ 방향 있음 |

---
### 실행 방법
1. **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. **셀을 위에서 아래로 순서대로 실행** (Shift+Enter)
3. 실제 데이터: 섹션 3에서 `USE_REAL_DATA = True` 설정 후 pkl 업로드

## 1. 패키지 설치 및 환경 설정

In [ ]:
!pip install timm -q

import os, random, json, time, pickle, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('준비 완료!')

## 2. 핵심 코드 정의

In [ ]:
# ──────────────────────────────────────────────────────────
# 상수
# ──────────────────────────────────────────────────────────
DEFECT_CLASSES = ['Center','Donut','Edge-Loc','Edge-Ring',
                  'Loc','Near-full','Random','Scratch']
DEFECT_COLORS  = ['#e74c3c','#e67e22','#f1c40f','#2ecc71',
                  '#1abc9c','#3498db','#9b59b6','#e91e63']
NUM_CLASSES = len(DEFECT_CLASSES)

# ImageNet 정규화 통계 (pretrained 모델용)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
NEUTRAL_MEAN  = [0.5, 0.5, 0.5]   # from-scratch CNN용
NEUTRAL_STD   = [0.5, 0.5, 0.5]

# ──────────────────────────────────────────────────────────
# 웨이퍼 맵 → RGB 이미지
# ──────────────────────────────────────────────────────────
def pad_to_square(wmap):
    h, w = wmap.shape
    s = max(h, w)
    out = np.zeros((s, s), dtype=wmap.dtype)
    ph, pw = (s-h)//2, (s-w)//2
    out[ph:ph+h, pw:pw+w] = wmap
    return out

def wafer_to_rgb(wmap, size=64):
    arr = np.array(Image.fromarray(pad_to_square(wmap.astype(np.uint8)))
                   .resize((size, size), Image.NEAREST), dtype=np.float32)
    rgb = np.zeros((size, size, 3), dtype=np.float32)
    rgb[arr == 0] = [0.0, 0.0, 0.0]   # 배경
    rgb[arr == 1] = [0.6, 0.6, 0.8]   # 정상 다이
    rgb[arr == 2] = [1.0, 0.2, 0.2]   # 불량 다이
    return rgb

print('상수 / 변환 함수 정의 완료')

In [ ]:
# ──────────────────────────────────────────────────────────
# [수정 1] Augmentation
#
# - RandomRotation(90)은 -90°~+90° 연속 회전 → 격자 구조 파괴
#   → 0/90/180/270° 중 하나만 선택하는 Rot90으로 교체
# - Edge-Loc, Scratch, Loc처럼 방향/위치가 클래스 정체성인
#   경우에도 90°단위 회전·뒤집기는 여전히 유효한 웨이퍼 패턴을
#   만들어주므로 사용합니다. (실 WM-811K 논문에서도 동일 사용)
# ──────────────────────────────────────────────────────────
class Rot90:
    """0 / 90 / 180 / 270 도 중 하나를 균등 확률로 선택"""
    def __call__(self, img):
        k = random.choice([0, 90, 180, 270])
        return TF.rotate(img, k)

# ──────────────────────────────────────────────────────────
# [수정 2] 정규화 — pretrained 여부에 따라 통계 분리
# ──────────────────────────────────────────────────────────
def make_transform(augment=False, pretrained=True):
    mean = IMAGENET_MEAN if pretrained else NEUTRAL_MEAN
    std  = IMAGENET_STD  if pretrained else NEUTRAL_STD
    ops = []
    if augment:
        ops += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            Rot90(),   # ← 연속 회전 대신 90° 단위 이산 회전
        ]
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
    return transforms.Compose(ops)

# ──────────────────────────────────────────────────────────
# Dataset
# ──────────────────────────────────────────────────────────
class WaferDataset(Dataset):
    def __init__(self, df, image_size=64, augment=False, pretrained=True):
        self.df        = df.reset_index(drop=True)
        self.labels    = self.df['label'].tolist()
        self.transform = make_transform(augment, pretrained)

        print(f'  이미지 변환 중... ({len(df)}개)', end='', flush=True)
        self.images = [
            Image.fromarray(
                (wafer_to_rgb(np.array(row['waferMap']), image_size)*255).astype(np.uint8)
            )
            for _, row in self.df.iterrows()
        ]
        print(' 완료!')

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return self.transform(self.images[idx]), self.labels[idx]

print('Dataset / Augmentation 정의 완료')

In [ ]:
# ──────────────────────────────────────────────────────────
# 모델
# ──────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                  nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                  nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                  nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        if pool: layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class WaferCNN(nn.Module):
    def __init__(self, num_classes=8, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(3,32), ConvBlock(32,64),
            ConvBlock(64,128), ConvBlock(128,256),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(256,256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes))
    def forward(self, x): return self.net(x)

class WaferResNet(nn.Module):
    def __init__(self, num_classes=8, pretrained=True, dropout=0.4):
        super().__init__()
        w = models.ResNet18_Weights.DEFAULT if pretrained else None
        net = models.resnet18(weights=w)
        # 64×64 소형 입력에 맞게 첫 레이어 조정 (stride 제거)
        net.conv1   = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        net.maxpool = nn.Identity()
        net.fc      = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, num_classes))
        self.net = net
    def forward(self, x): return self.net(x)

class WaferEfficientNet(nn.Module):
    def __init__(self, num_classes=8, pretrained=True, dropout=0.4):
        super().__init__()
        w = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        net = models.efficientnet_b0(weights=w)
        net.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(1280, num_classes))
        self.net = net
    def forward(self, x): return self.net(x)

def build_model(name='resnet18', pretrained=True, num_classes=8, dropout=0.4):
    if   name == 'cnn':         return WaferCNN(num_classes, dropout)
    elif name == 'resnet18':    return WaferResNet(num_classes, pretrained, dropout)
    elif name == 'efficientnet':return WaferEfficientNet(num_classes, pretrained, dropout)
    else: raise ValueError(f'알 수 없는 모델: {name}')

print('모델 정의 완료')

In [ ]:
# ──────────────────────────────────────────────────────────
# [수정 3] Early Stopping 추가
# ──────────────────────────────────────────────────────────
def run_training(model, train_loader, val_loader,
                 epochs=30, lr=0.001, patience=7):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    os.makedirs('/content/checkpoints', exist_ok=True)
    best_acc, best_state, no_improve = 0.0, None, 0
    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

    pbar = tqdm(range(1, epochs+1), desc='Training')
    for epoch in pbar:
        # ── train
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward(); optimizer.step()
            tl += loss.item()*imgs.size(0)
            tc += (out.argmax(1)==labels).sum().item()
            tn += imgs.size(0)

        # ── val
        model.eval()
        vl, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out  = model(imgs)
                vl  += criterion(out, labels).item()*imgs.size(0)
                vc  += (out.argmax(1)==labels).sum().item()
                vn  += imgs.size(0)

        tr_loss, tr_acc = tl/tn, tc/tn
        vl_loss, vl_acc = vl/vn, vc/vn
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        pbar.set_postfix({'tr': f'{tr_acc:.3f}', 'val': f'{vl_acc:.3f}',
                          'patience': f'{no_improve}/{patience}'})

        # ── checkpoint & early stopping
        if vl_acc > best_acc:
            best_acc   = vl_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            torch.save(best_state, '/content/checkpoints/best_model.pt')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'\n  Early stopping at epoch {epoch} (best val_acc={best_acc:.4f})')
                break

    print(f'\n최고 val accuracy: {best_acc:.4f}')
    model.load_state_dict(best_state)  # best 모델 복원
    return history

print('학습 함수 (Early Stopping 포함) 정의 완료')

In [ ]:
# ──────────────────────────────────────────────────────────
# 시각화
# ──────────────────────────────────────────────────────────
def plot_class_dist(df):
    counts = df['label_name'].value_counts()
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = [DEFECT_COLORS[DEFECT_CLASSES.index(c)] for c in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white')
    ax.bar_label(bars, padding=3)
    ax.set_title('클래스 분포', fontsize=14, fontweight='bold')
    ax.set_xlabel('결함 유형'); ax.set_ylabel('샘플 수')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()

def plot_samples(df, n=4, size=64):
    classes = sorted(df['label_name'].unique())
    fig, axes = plt.subplots(len(classes), n, figsize=(n*2.5, len(classes)*2.5))
    for r, cls in enumerate(classes):
        sub = df[df['label_name']==cls].sample(min(n, sum(df['label_name']==cls)),
                                               random_state=SEED)
        for c in range(n):
            ax = axes[r, c]
            if c < len(sub):
                ax.imshow(wafer_to_rgb(np.array(sub.iloc[c]['waferMap']), size))
            ax.axis('off')
            if c == 0: ax.set_title(cls, fontsize=9, fontweight='bold')
    plt.suptitle('웨이퍼 맵 샘플 (빨강=불량, 파란빛=정상)', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

def plot_curves(history):
    ep = range(1, len(history['train_loss'])+1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    a1.plot(ep, history['train_loss'], label='Train', color='#3498db', lw=2)
    a1.plot(ep, history['val_loss'],   label='Val',   color='#e74c3c', lw=2)
    a1.set_title('Loss', fontweight='bold'); a1.set_xlabel('Epoch')
    a1.legend(); a1.grid(alpha=0.3)
    a2.plot(ep, [x*100 for x in history['train_acc']], label='Train', color='#3498db', lw=2)
    a2.plot(ep, [x*100 for x in history['val_acc']],   label='Val',   color='#e74c3c', lw=2)
    a2.set_title('Accuracy (%)', fontweight='bold'); a2.set_xlabel('Epoch')
    a2.legend(); a2.grid(alpha=0.3)
    plt.suptitle('학습 곡선', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

def plot_confusion(labels, preds):
    cm = confusion_matrix(labels, preds).astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=DEFECT_CLASSES, yticklabels=DEFECT_CLASSES, ax=ax)
    ax.set_title('혼동 행렬 (행=실제 클래스, 대각선=정답률)', fontsize=12, fontweight='bold')
    ax.set_xlabel('예측'); ax.set_ylabel('실제')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

@torch.no_grad()
def plot_predictions(model, dataset, n=16):
    model.eval()
    idxs = random.sample(range(len(dataset)), n)
    fig, axes = plt.subplots(4, 4, figsize=(12, 13))
    for ax, idx in zip(axes.flatten(), idxs):
        tensor, true = dataset[idx]
        logit = model(tensor.unsqueeze(0).to(device))
        pred  = logit.argmax(1).item()
        conf  = torch.softmax(logit, 1).max().item()
        img   = np.clip(tensor.permute(1,2,0).numpy()*0.5+0.5, 0, 1)
        ax.imshow(img)
        color = 'limegreen' if pred==true else 'red'
        ax.set_title(f'정답: {DEFECT_CLASSES[true]}\n'
                     f'예측: {DEFECT_CLASSES[pred]} ({conf:.0%})',
                     fontsize=8, color=color)
        ax.axis('off')
    plt.suptitle('예측 결과 (초록=정답, 빨강=오답)', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

print('시각화 함수 정의 완료')

## 3. 데이터 로드

### ✅ 옵션 A — 합성 데이터 (바로 실행 가능)
> ⚠️ **주의:** 합성 데이터는 패턴이 너무 규칙적이라 90%+ 나와도 실제 데이터 성능과 무관합니다.  
> 파이프라인 점검 용도로만 사용하세요.

### ⬆️ 옵션 B — 실제 WM-811K 데이터 (권장)
> https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map 에서 `LSWMD.pkl` 다운로드 후 업로드

In [ ]:
# 합성 데이터 생성기
def _base(size=26):
    m = np.zeros((size,size), dtype=np.uint8)
    cx, cy, r = size//2, size//2, size//2-1
    for i in range(size):
        for j in range(size):
            if (i-cx)**2+(j-cy)**2<=r**2: m[i,j]=1
    return m

def _gen(pattern, size=26, rng=None):
    rng = rng or np.random.default_rng()
    m = _base(size).copy()
    cx, cy, r = size//2, size//2, size//2-1
    if pattern=='Center':
        cr=int(r*0.35)
        for i in range(size):
            for j in range(size):
                if m[i,j]==1 and (i-cx)**2+(j-cy)**2<=cr**2 and rng.random()<0.7: m[i,j]=2
    elif pattern=='Donut':
        inn,out=int(r*0.35),int(r*0.75)
        for i in range(size):
            for j in range(size):
                d2=(i-cx)**2+(j-cy)**2
                if m[i,j]==1 and inn**2<=d2<=out**2 and rng.random()<0.6: m[i,j]=2
    elif pattern=='Edge-Ring':
        inn=int(r*0.75)
        for i in range(size):
            for j in range(size):
                if m[i,j]==1 and (i-cx)**2+(j-cy)**2>=inn**2 and rng.random()<0.7: m[i,j]=2
    elif pattern=='Edge-Loc':
        angle=rng.uniform(0,2*np.pi)
        for i in range(size):
            for j in range(size):
                if m[i,j]!=1: continue
                d2=(i-cx)**2+(j-cy)**2
                a=np.arctan2(i-cx,j-cy)
                if d2>=(r*0.7)**2 and abs(a-angle)<0.6 and rng.random()<0.7: m[i,j]=2
    elif pattern=='Loc':
        lx,ly,lr=rng.integers(2,size-3),rng.integers(2,size-3),rng.integers(2,5)
        for i in range(size):
            for j in range(size):
                if m[i,j]==1 and (i-lx)**2+(j-ly)**2<=lr**2 and rng.random()<0.75: m[i,j]=2
    elif pattern=='Scratch':
        x0,y0=rng.integers(2,size-2),rng.integers(2,size-2)
        angle,length=rng.uniform(0,np.pi),rng.integers(6,size-2)
        for t in range(length):
            xi,yi=int(x0+t*np.sin(angle)),int(y0+t*np.cos(angle))
            if 0<=xi<size and 0<=yi<size and m[xi,yi]==1:
                m[xi,yi]=2
                for di in [-1,0,1]:
                    ni=xi+di
                    if 0<=ni<size and m[ni,yi]==1 and rng.random()<0.4: m[ni,yi]=2
    elif pattern=='Random':
        for i in range(size):
            for j in range(size):
                if m[i,j]==1 and rng.random()<0.08: m[i,j]=2
    elif pattern=='Near-full':
        for i in range(size):
            for j in range(size):
                if m[i,j]==1 and rng.random()<0.85: m[i,j]=2
    return m

def make_synthetic_df(n_per_class=400, size=26):
    rng  = np.random.default_rng(SEED)
    rows = []
    for ci, cls in enumerate(DEFECT_CLASSES):
        for _ in range(n_per_class):
            rows.append({'waferMap': _gen(cls, size, rng),
                         'label_name': cls, 'label': ci})
    return pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

print('합성 데이터 생성기 준비')

In [ ]:
# ══════════════════════════════════════════════════════════
# 여기서 선택
#   False → 합성 데이터 (다운로드 불필요, 파이프라인 확인용)
#   True  → 실제 WM-811K (Kaggle에서 LSWMD.pkl 다운로드 필요)
# ══════════════════════════════════════════════════════════
USE_REAL_DATA = False

if not USE_REAL_DATA:
    print('합성 데이터 생성 중... (클래스 8개 × 400개 = 3200개)')
    df = make_synthetic_df(n_per_class=400)
    print('생성 완료!')
    print('\n⚠️  합성 데이터는 너무 규칙적이라 점수가 높게 나옵니다.')
    print('   실제 성능 평가는 반드시 USE_REAL_DATA = True 로 해주세요.\n')

else:
    # ── [수정 4] 실제 데이터 파싱 — 구조 자동 감지
    from google.colab import files
    print('LSWMD.pkl 업로드하세요...')
    uploaded = files.upload()

    with open('LSWMD.pkl', 'rb') as f:
        raw_df = pickle.load(f)

    # ── failureType 구조 확인 (디버그)
    sample_ft = raw_df['failureType'].dropna().iloc[0]
    print(f'failureType 샘플값: {sample_ft}  |  type: {type(sample_ft)}')
    if hasattr(sample_ft, '__len__') and len(sample_ft) > 0:
        inner = sample_ft[0]
        print(f'  내부 요소: {inner}  |  type: {type(inner)}')

    # ── 파싱 (numpy array / list 두 가지 구조 모두 처리)
    def parse_failure_type(x):
        try:
            if hasattr(x, '__len__') and len(x) > 0:
                inner = x[0]
                if hasattr(inner, '__len__') and not isinstance(inner, str):
                    return str(inner[0])   # [['Center']] 형태
                return str(inner)          # ['Center'] 형태
        except Exception:
            pass
        return None

    raw_df = raw_df.dropna(subset=['failureType'])
    raw_df = raw_df[raw_df['failureType'].apply(
        lambda x: hasattr(x, '__len__') and len(x) > 0)]
    raw_df['label_name'] = raw_df['failureType'].apply(parse_failure_type)
    raw_df = raw_df.dropna(subset=['label_name'])
    raw_df = raw_df[raw_df['label_name'].isin(DEFECT_CLASSES)]

    label_map = {c: i for i, c in enumerate(DEFECT_CLASSES)}
    raw_df['label'] = raw_df['label_name'].map(label_map).astype(int)
    df = raw_df.reset_index(drop=True)
    print(f'\n실제 데이터 로드 완료: {len(df)}개')

print(f'총 샘플 수: {len(df)}')
print(df['label_name'].value_counts().to_string())

## 4. 탐색적 데이터 분석 (EDA)

In [ ]:
plot_class_dist(df)

In [ ]:
plot_samples(df, n=4)

## 5. 데이터 분할 및 DataLoader

In [ ]:
IMAGE_SIZE = 64
BATCH_SIZE = 128
MODEL_NAME = 'resnet18'  # 'cnn' | 'resnet18' | 'efficientnet'
PRETRAINED = True

# stratified split
train_df, temp_df = train_test_split(df, test_size=0.3,
                                     stratify=df['label'], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.5,
                                     stratify=temp_df['label'], random_state=SEED)

print('이미지 준비 중...')
# [수정 2] pretrained 여부를 Dataset에 전달 → 정규화 통계 자동 선택
train_ds = WaferDataset(train_df, IMAGE_SIZE, augment=True,  pretrained=PRETRAINED)
val_ds   = WaferDataset(val_df,   IMAGE_SIZE, augment=False, pretrained=PRETRAINED)
test_ds  = WaferDataset(test_df,  IMAGE_SIZE, augment=False, pretrained=PRETRAINED)

# 클래스 불균형 처리 — Weighted Sampler
counts   = np.bincount(train_df['label'].tolist())
w_class  = 1.0 / counts
w_sample = [w_class[l] for l in train_df['label'].tolist()]
sampler  = WeightedRandomSampler(w_sample, len(w_sample), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'\ntrain: {len(train_df)}개 | val: {len(val_df)}개 | test: {len(test_df)}개')
print(f'정규화: {"ImageNet 통계" if PRETRAINED else "[0.5,0.5,0.5] (from-scratch)"}')

## 6. 모델 선택 및 학습

위 셀의 `MODEL_NAME`을 바꾸면 모델이 달라집니다.

| 설정 | 특징 |
|---|---|
| `'cnn'` + `PRETRAINED=False` | 빠름, 경량 |
| `'resnet18'` + `PRETRAINED=True` | 균형 좋음 ← 추천 |
| `'efficientnet'` + `PRETRAINED=True` | 파라미터 효율 좋음 |

In [ ]:
EPOCHS  = 40
LR      = 0.001
PATIENCE = 8   # val_acc가 이 epoch 수 동안 개선 없으면 중단

model = build_model(MODEL_NAME, PRETRAINED, NUM_CLASSES).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'모델: {MODEL_NAME.upper()}  |  파라미터: {n_params:,}  |  pretrained: {PRETRAINED}')

history = run_training(model, train_loader, val_loader,
                       epochs=EPOCHS, lr=LR, patience=PATIENCE)

## 7. 결과 분석

In [ ]:
plot_curves(history)

In [ ]:
# ── 예측 수집
@torch.no_grad()
def get_preds(model, loader):
    model.eval()
    preds, labels = [], []
    for imgs, lbs in loader:
        out = model(imgs.to(device))
        preds.extend(out.argmax(1).cpu().tolist())
        labels.extend(lbs.tolist())
    return np.array(preds), np.array(labels)

preds, labels = get_preds(model, test_loader)

# ── [수정 5] 지표 출력 강화
acc = (preds == labels).mean()
f1_macro    = f1_score(labels, preds, average='macro')
f1_weighted = f1_score(labels, preds, average='weighted')

print('=' * 55)
print(f'  정확도 (Accuracy)   : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  F1 macro            : {f1_macro:.4f}   ← 불균형 데이터 핵심 지표')
print(f'  F1 weighted         : {f1_weighted:.4f}')
print('=' * 55)
print('\n클래스별 성능 (recall이 낮은 클래스에 집중):')
print(classification_report(labels, preds, target_names=DEFECT_CLASSES))

In [ ]:
plot_confusion(labels, preds)

In [ ]:
plot_predictions(model, test_ds)

## 8. 모델 저장 / 다운로드

In [ ]:
save_path = f'/content/wafer_{MODEL_NAME}_final.pt'
torch.save(model.state_dict(), save_path)
print(f'저장 완료: {save_path}')

from google.colab import files
files.download(save_path)